# 엑셀/PDF 파일 읽기 실습

- 엑셀 파일은 `openpyxl`, PDF 파일은 `pypdf` 패키지를 사용


## 패키지 설치

터미널에서 아래 명령어를 한 번 실행한다.

```bash
python -m pip install openpyxl pypdf
```


## 엑셀 파일 읽기

엑셀 파일은 단순 문자열 파일이 아니라 여러 시트, 셀, 서식 정보가 들어 있는 구조화 파일이다. 그래서 `openpyxl` 같은 전용 라이브러리로 읽는다.


In [1]:
# load_workbook(): 엑셀을 열어서 Workbook 객체로 변환
from openpyxl import load_workbook

# data_only=True: 셀에 작성된 데이터 읽어오기(수식 X, 수식 결과 O)
workbook = load_workbook('students.xlsx', data_only=True)

print(workbook.sheetnames)  # 엑셀의 모든 시트 읽어오기

worksheet = workbook['students']  # 'students' 시트만 얻어오기

print(worksheet.title)  # 시트 이름('students')
print('worksheet.max_row: ', worksheet.max_row)
print('worksheet.max_column: ', worksheet.max_column)

['students']
students
worksheet.max_row:  5
worksheet.max_column:  5


In [2]:
# 특정 셀 값 읽어오기
print('B2 값: ', worksheet['B2'].value)
print('D3 값: ', worksheet['D3'].value)
print('4행 2열 값: ', worksheet.cell(row=4, column=2).value)

B2 값:  홍길동
D3 값:  85
4행 2열 값:  유관순


In [3]:
# 엑셀 한 줄 씩 읽기
# iter == iterator == 반복자
# iter_rows == 행 반복자: 반복 될 때 마다 다음 행 반환
# values_only=True: 셀 안의 실제 값만 tuple로 가져오기
for row in worksheet.iter_rows(values_only=True):
    print(row)

('id', 'name', 'course', 'score', 'passed')
(1, '홍길동', 'Python IO', 92, True)
(2, '이순신', 'Python IO', 85, True)
(3, '유관순', 'Python IO', 78, True)
(4, '신사임당', 'Python IO', 64, False)


In [74]:
# 엑셀 행 데이터를 dict list로 변환
rows = list(worksheet.iter_rows(values_only=True))
print(rows)

# 0행: ('id', 'name', 'course', 'score', 'passed')
# 1~4행: (1, '홍길동', 'Python IO', 92, True)

headers = rows[0]  # 컬럼명 -> dict key로 사용

student_list: list[dict] = []  # dict list 결과 저장용 변수

for row in rows[1:]:
    # 학생 dict 데이터 생성
    # zip(tuple, tuple) -> 두 튜플의 같은 인덱스 요소끼리 K:V 쌍으로 묶어서 dict 반환
    student = dict(zip(headers, row))
    student_list.append(student)

for student in student_list:
    print(student, type(student))

[('id', 'name', 'course', 'score', 'passed'), (1, '홍길동', 'Python IO', 92, True), (2, '이순신', 'Python IO', 85, True), (3, '유관순', 'Python IO', 78, True), (4, '신사임당', 'Python IO', 64, False)]
{'id': 1, 'name': '홍길동', 'course': 'Python IO', 'score': 92, 'passed': True} <class 'dict'>
{'id': 2, 'name': '이순신', 'course': 'Python IO', 'score': 85, 'passed': True} <class 'dict'>
{'id': 3, 'name': '유관순', 'course': 'Python IO', 'score': 78, 'passed': True} <class 'dict'>
{'id': 4, 'name': '신사임당', 'course': 'Python IO', 'score': 64, 'passed': False} <class 'dict'>


In [73]:
# 엑셀에서 읽어온 학생 데이터로 학생수, 점수 합계, 평균, 최고점, 최저점 구하기

# list[dict] -> dict -> dict[score]만 뽑아서 list[int] 생성
scores: list[int] = [student['score'] for student in student_list]
sum_score = sum(scores)
avg_score = sum_score / len(student_list)
max_score = max(scores)
min_score = min(scores)

print("학생 수: ", len(student_list))
print("점수 합계: ", sum_score)
print("점수 평균: ", avg_score)
print("최고 점수: ", max_score)
print("최저 점수: ", min_score)

학생 수:  4
점수 합계:  319
점수 평균:  79.75
최고 점수:  92
최저 점수:  64


## PDF 파일 읽기

- PDF는 페이지, 글자 위치, 폰트, 이미지 정보가 함께 들어 있는 문서 파일이다. 그래서 텍스트 파일처럼 바로 읽기보다 `pypdf` 같은 라이브러리로 페이지 단위 텍스트를 추출한다.

- 한글 PDF는 폰트와 인코딩 방식에 따라 텍스트 추출 결과가 달라질 수 있음



In [78]:
# pypdf: 파이썬에서 pdf 파일을 읽고, 쓸 수 있게 하는 라이브러리(모듈)
from pypdf import PdfReader

# pathlib.Path: 파일/폴더 경로를 편하게 다루기 위한 클래스
# 현재 노트북파일이 실행되는 위치를 기준으로 한다(기본 값)
from pathlib import Path
pdf_path = Path('io_sample.pdf')
# print(pdf_path)

reader = PdfReader(pdf_path)
# print(reader, type(reader))

# reader.pages: 현재 읽어들인 pdf 파일의 페이지 객체를 담아둔 list
print("reader.pages: ", reader.pages)
print("읽은 pdf의 page 수: ", len(reader.pages))

incorrect startxref pointer(1)
parsing for Object Streams


reader.pages:  [PageObject(0), PageObject(1)]
읽은 pdf의 page 수:  2


In [79]:
# 첫 번째 페이지 text 추출
first_page_text = reader.pages[0].extract_text()
print("첫 번째 페이지 text: ", first_page_text)

첫 번째 페이지 text:  IO Practice Report
This PDF is used for Python file I/O practice.
PDF files are structured documents, not plain text files.
We use pypdf to extract text page by page.



In [80]:
# 전체 페이지 text 추출
# 단, 페이지 별로 번호를 이용해 구분

for page_number, page in enumerate(reader.pages, start=1):
    print(f'--- page_number: {page_number} ---')
    print(page.extract_text())

--- page_number: 1 ---
IO Practice Report
This PDF is used for Python file I/O practice.
PDF files are structured documents, not plain text files.
We use pypdf to extract text page by page.

--- page_number: 2 ---
Second Page
A PDF can contain multiple pages.
In Python, each page can be read and processed separately.

